In [1]:
import os
import gc
import time
import random
import pickle
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler

# Torchvision
from torchvision import models
from torchvision import transforms

# Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

# Visualization
import matplotlib.pyplot as plt

print("Imports successful.")

Imports successful.


In [2]:
# DEVICE SETUP

torch.set_num_threads(os.cpu_count())

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("=" * 50)
    print("GPU DETECTED")
    print("=" * 50)

    print("GPU Name:")
    print(torch.cuda.get_device_name(0))

    print("\nCUDA Version:")
    print(torch.version.cuda)

else:

    device = torch.device("cpu")

    print("=" * 50)
    print("NO CUDA GPU DETECTED")
    print("=" * 50)

    print("Using CPU.")

NO CUDA GPU DETECTED
Using CPU.


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seeds initialized.")

Seeds initialized.


In [4]:
# =========================================================
# CELL 4 — DATASET PATHS
# LOCAL OR KAGGLE
# =========================================================

USE_KAGGLE = False

# =========================================================
# LOCAL PATHS
# =========================================================

LOCAL_DATA_DIR = "D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed"

# =========================================================
# KAGGLE PATHS
# CHANGE DATASET NAME LATER
# =========================================================

KAGGLE_DATA_DIR = "/kaggle/input/YOUR_DATASET_NAME/processed"

# =========================================================
# SELECT PATH
# =========================================================

if USE_KAGGLE:

    DATA_DIR = KAGGLE_DATA_DIR

else:

    DATA_DIR = LOCAL_DATA_DIR

# =========================================================
# PKL FILES
# =========================================================

TRAIN_PKL = os.path.join(DATA_DIR, "wlasl_train.pkl")

VAL_PKL = os.path.join(DATA_DIR, "wlasl_val.pkl")

TEST_PKL = os.path.join(DATA_DIR, "wlasl_test.pkl")

print(TRAIN_PKL)
print(VAL_PKL)
print(TEST_PKL)

D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed\wlasl_train.pkl
D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed\wlasl_val.pkl
D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed\wlasl_test.pkl


In [5]:
# CONFIG
CONFIG = {

    "batch_size": 8 if torch.cuda.is_available() else 2,

    "epochs": 3,

    "learning_rate": 1e-4,

    "weight_decay": 1e-4,

    "dropout": 0.3,

    "hidden_size": 128,

    "lstm_layers": 2,

    "gradient_clip": 1.0,

    "early_stopping_patience": 5,

    "scheduler_patience": 2,

    "subset_size": 500,

    "freeze_cnn": True,

    "num_workers": 2 if torch.cuda.is_available() else 0
}

print(CONFIG)

{'batch_size': 2, 'epochs': 3, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'dropout': 0.3, 'hidden_size': 128, 'lstm_layers': 2, 'gradient_clip': 1.0, 'early_stopping_patience': 5, 'scheduler_patience': 2, 'subset_size': 500, 'freeze_cnn': True, 'num_workers': 0}


In [6]:
# =========================================================
# CELL 6 — AUGMENTATION
# =========================================================

class VideoAugmentation:

    def __init__(self):

        self.transform = transforms.Compose([

            transforms.ToPILImage(),

            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2
            ),

            transforms.RandomRotation(5),

            transforms.ToTensor()
        ])

    def __call__(self, frames):

        augmented_frames = []

        for frame in frames:

            frame = self.transform(frame)

            augmented_frames.append(frame)

        return torch.stack(augmented_frames)

In [7]:
# =========================================================
# CELL 7 — DATASET CLASS
# =========================================================

class SignLanguageDataset(Dataset):

    def __init__(
        self,
        pkl_path,
        train=False,
        subset_size=None
    ):

        with open(pkl_path, "rb") as f:

            data = pickle.load(f)

        self.frames = data["frames"]
        self.labels = data["labels"]

        # =================================================
        # RANDOM SUBSET
        # =================================================

        if subset_size is not None:

            subset_size = min(subset_size, len(self.labels))

            indices = np.random.choice(
                len(self.labels),
                subset_size,
                replace=False
            )

            self.frames = self.frames[indices]
            self.labels = self.labels[indices]

        self.train = train

        self.augment = VideoAugmentation()

        print(f"Loaded {len(self.labels)} samples.")

        print("Unique classes:",
              len(np.unique(self.labels)))

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        video = self.frames[idx]

        # [T,H,W,C] -> [T,C,H,W]

        video = torch.tensor(video).permute(0,3,1,2).float()

        # Normalize

        video = video / 255.0

        # Augmentation

        if self.train:

            video = self.augment(video)

        label = torch.tensor(
            self.labels[idx]
        ).long()

        return video, label

In [8]:
# =========================================================
# CELL 8 — LOAD DATASETS
# =========================================================

train_dataset = SignLanguageDataset(
    TRAIN_PKL,
    train=True,
    subset_size=CONFIG["subset_size"]
)

val_dataset = SignLanguageDataset(
    VAL_PKL,
    train=False,
    subset_size=200
)

test_dataset = SignLanguageDataset(
    TEST_PKL,
    train=False,
    subset_size=200
)

Loaded 500 samples.
Unique classes: 1
Loaded 200 samples.
Unique classes: 1
Loaded 200 samples.
Unique classes: 1
